In [ ]:
import pandas as pd
from urllib.request import urlopen, Request
from bs4 import BeautifulSoup
from pathlib import Path
import datetime

# ==========================================
# 1. SETUP PATHS
# ==========================================
REQUIRED_FOLDERS = ['data', 'code', 'models', 'images', 'output']
def get_project_root():
    current_path = Path.cwd()
    if current_path.name in REQUIRED_FOLDERS:
        return current_path.parent
    return current_path

BASE_PATH = get_project_root()
DATA_PATH = BASE_PATH / 'data'

# ==========================================
# 2. FINVIZ NEWS SCRAPER
# ==========================================
# FinViz blocks automated requests often, so we need a "User-Agent" header 
# to look like a real browser.
finviz_url = 'https://finviz.com/quote.ashx?t=SPY'

print(f"Scraping News from: {finviz_url}")

req = Request(url=finviz_url, headers={'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/58.0.3029.110 Safari/537.3'})

try:
    response = urlopen(req)
    html = BeautifulSoup(response, 'html.parser')
    print("Connection Successful. Parsing HTML...")
except Exception as e:
    print(f"Connection Failed: {e}")
    # If this fails, we will need to discuss an alternative static file approach.

# ==========================================
# 3. PARSING THE NEWS TABLE
# ==========================================
news_parsed = []

# FinViz usually stores news in a table with id="news-table"
news_table = html.find(id='news-table')

if news_table:
    # Iterate through all table rows <tr>
    for row in news_table.findAll('tr'):
        
        # The text is inside an anchor <a> tag usually
        text_element = row.a 
        if not text_element:
            continue
            
        headline = text_element.get_text()
        link = text_element['href']
        
        # Timestamp is in the <td> text, separate from the link
        timestamp_data = row.td.get_text().split()
        
        # FinViz format is usually: "Jan-18-24 09:30PM" or just "09:30PM" if same day
        if len(timestamp_data) == 1:
            time = timestamp_data[0]
            # Date remains the same as the last iteration, so we don't update 'date'
        else:
            date = timestamp_data[0]
            time = timestamp_data[1]
        
        news_parsed.append([date, time, headline])
        
    print(f"Extracted {len(news_parsed)} headlines.")
else:
    print("Could not find news table. FinViz structure might have changed.")

# ==========================================
# 4. DATAFRAME CREATION & CLEANING
# ==========================================
df_news = pd.DataFrame(news_parsed, columns=['date', 'time', 'headline'])

# Handle "Today" or "Yesterday" if Finviz uses them (usually they use MMM-DD-YY)
# We need to convert date strings to Datetime objects.
# Note: FinViz dates are usually "Jan-18-24".
print("\n--- Raw News Preview ---")
print(df_news.head())

# Save Raw News
news_path = DATA_PATH / 'news_raw_finviz.csv'
df_news.to_csv(news_path, index=False)
print(f"Raw News saved to: {news_path}")

🕵️‍♂️ Scraping News from: https://finviz.com/quote.ashx?t=SPY
✅ Connection Successful. Parsing HTML...
✅ Extracted 100 headlines.

--- Raw News Preview ---
        date     time                                           headline
0  Jan-17-26  05:35PM  Here's How Much $1,000 in a Trump Account Coul...
1  Jan-17-26  10:33AM  Investors Worried About Large Cap Concentratio...
2  Jan-16-26  08:50PM    VOO vs. SPY: What's the Better S&P 500 ETF Buy?
3  Jan-16-26  04:38PM  Stocks Finish Slightly Lower as Bond Yields Climb
4  Jan-16-26  04:29PM  Top 10 Congress Stock Traders 2025: Nancy Pelo...
💾 Raw News saved to: c:\Users\Yahya\Desktop\My folder\WQU\10. Capstone\Thesis Work\Machine Learning & Deep Learning in Finance\NLP for Intraday Volatility Prediction\data\news_raw_finviz.csv
